# Analysis routine to produce figures
## To run this code, you will need the output files below:

**main files**: test_rank0_iter???_mu_t_list_all.npz <br>
**ancillary files**: test_rank0_iter???_model_PhiDP_offset_long, test_rank0_iter???_model_PhiDP_offset_short, test_rank0_iter???_angle_offset.txt, test_rank0_iter???_maskall.npz, test_rank0_iter???_obsdata.npz

## adjust the parameters below to the path where you save your data.

In [ ]:
FOLDER = f"../../MPPAWR_data/250313_test/cond1/" # Directory for the NN output: files such as test_rank0_iter???_mu_t_list_all.npz
path_datetime_Kumagaya = "../txtfiles_datetime/202206_datetime_Kumagaya.txt" # Kumagaya radar datetime when Kumagaya is raining
lookup_tables_path = "../lookup_tables" # Directory for the lookup tables
OUTDIR = "figures_Kumagaya_newexperiment_250312"

In [ ]:
import os
from einops import rearrange
import numpy as np
import torch
from datetime import datetime
import matplotlib.pyplot as plt

datetime_Kumagaya = np.loadtxt(path_datetime_Kumagaya, dtype=str)

def output_coordinate(angle_offset, el_angle):
    # we will output a 300*800 tensor that contains coordinate information
    X = np.zeros((300, 800))
    Y = np.zeros((300, 800))
    angles = np.array([90-(angle_offset+1.2*i) for i in range(300)])
    for i in range(800):
        X[:, i] = (i+1)*0.075*np.cos(angles*np.pi/180)*np.cos(el_angle*np.pi/180)
        Y[:, i] = (i+1)*0.075*np.sin(angles*np.pi/180)*np.cos(el_angle*np.pi/180)
        
    return X, Y

def load_lookups():
    table_mu_AH = torch.tensor(np.loadtxt(f"{lookup_tables_path}/lognorm_AH_el5.txt"), device="cpu", dtype=torch.float32)
    table_mu_AV = torch.tensor(np.loadtxt(f"{lookup_tables_path}/lognorm_AV_el5.txt"), device="cpu", dtype=torch.float32)
    table_mu_kdp = torch.tensor(np.loadtxt(f"{lookup_tables_path}/lognorm_kdp_el5.txt"), device="cpu", dtype=torch.float32)
    table_mu_ZH = torch.tensor(np.loadtxt(f"{lookup_tables_path}/lognorm_ZH_el5.txt"), device="cpu", dtype=torch.float32)
    table_mu_Zdr = torch.tensor(np.loadtxt(f"{lookup_tables_path}/lognorm_Zdr_el5.txt"), device="cpu", dtype=torch.float32)
    return [table_mu_AH*(0.075*2), table_mu_AV*(0.075*2), table_mu_kdp*(0.075*2), table_mu_ZH, table_mu_Zdr]

def torch_interp1d(x, xp, yp):                                                                                                              
    idx_left = torch.searchsorted(xp, x, right=True) - 1
    idx_right = idx_left + 1
    slopes = (yp[idx_right] - yp[idx_left]) / (xp[idx_right] - xp[idx_left])
    interpolated = yp[idx_left] + slopes * (x - xp[idx_left])
    return interpolated

def torch_interp2d(x, y, xp, yp, zp):                                                                                                    
    idx_left = torch.searchsorted(xp, x, right=True) - 1
    idx_right = idx_left + 1
    idx_lower = torch.searchsorted(yp, y, right=True) - 1
    idx_upper = idx_lower + 1
    w_x = (x-xp[idx_left])/(xp[idx_right]-xp[idx_left])
    w_y = (y-yp[idx_lower])/(yp[idx_upper]-yp[idx_lower])
    interpolated = w_x*w_y*zp[idx_right, idx_upper] + (1-w_x)*w_y*zp[idx_left, idx_upper] + w_x*(1-w_y)*zp[idx_right, idx_lower] + (1-w_x)*(1-w_y)*zp[idx_left, idx_lower]
    return interpolated

def Maki(x):
    x_cm = x * 0.1
    # Apply conditions for both cases
    condition = (x_cm < 0.11) | (x_cm > 0.44)
    # Define the two cases
    result_case1 = 1.0048 + 0.0057*x_cm - 2.628*x_cm**2 + 3.682*x_cm**3 - 1.677*x_cm**4
    result_case2 = 1.0048 + 0.0057*x_cm - 2.628*x_cm**2 + 3.682*x_cm**3 - 1.677*x_cm**4
    #result_case2 = 1.012 - 0.144*x_cm - 1.03*x_cm**2
    return torch.where(condition, result_case1, result_case2)

def kDP_function(z, N0_range, loc_range, log_scale_range, lookup_tables):
    [table_mu_AH, table_mu_AV, table_mu_kdp, table_mu_ZH, table_mu_Zdr] = lookup_tables
    N0s = torch.sigmoid(z[..., 0]/(N0_range[1]-N0_range[0]))*(N0_range[1]-N0_range[0])+N0_range[0]
    locs = torch.sigmoid(z[..., 1]/(loc_range[1]-loc_range[0]))*(loc_range[1]-loc_range[0])+loc_range[0]
    log_scales = torch.sigmoid(z[..., 2]/(log_scale_range[1]-log_scale_range[0]))*(log_scale_range[1]-log_scale_range[0])+log_scale_range[0]
    xp = torch.linspace(-1.2, 1.2, 61, device=z.device)
    yp = torch.linspace(-0.2, 0.2, 51, device=z.device)
    kDP = torch_interp2d(x=locs, y=log_scales, xp=xp, yp=yp, zp=table_mu_kdp) * pow(10, N0s)
    return kDP

def observation_DSD_Zh(z, N0_range, loc_range, log_scale_range, lookup_tables):
    [table_mu_AH, table_mu_AV, table_mu_kdp, table_mu_ZH, table_mu_Zdr] = lookup_tables
    CZH = 0
    # assuming z contains 3 numbers per ot. 3 numbers are ZH, ZV, KDP                                                                                                                            
    N0s = torch.sigmoid(z[..., 0]/(N0_range[1]-N0_range[0]))*(N0_range[1]-N0_range[0])+N0_range[0]
    locs = torch.sigmoid(z[..., 1]/(loc_range[1]-loc_range[0]))*(loc_range[1]-loc_range[0])+loc_range[0]
    log_scales = torch.sigmoid(z[..., 2]/(log_scale_range[1]-log_scale_range[0]))*(log_scale_range[1]-log_scale_range[0])+log_scale_range[0]
    
    xp = torch.linspace(-1.2, 1.2, 61, device=z.device)
    yp = torch.linspace(-0.2, 0.2, 51, device=z.device)
    Zh = torch_interp2d(x=locs, y=log_scales, xp=xp, yp=yp, zp=table_mu_ZH) + 10*N0s
    Zdr = torch_interp2d(x=locs, y=log_scales, xp=xp, yp=yp, zp=table_mu_Zdr)
    kDP = torch_interp2d(x=locs, y=log_scales, xp=xp, yp=yp, zp=table_mu_kdp) * pow(10, N0s)
    AH = torch_interp2d(x=locs, y=log_scales, xp=xp, yp=yp, zp=table_mu_AH) * pow(10, N0s)
    AV = torch_interp2d(x=locs, y=log_scales, xp=xp, yp=yp, zp=table_mu_AV) * pow(10, N0s)
    
    PhiDP = z[..., 3]                                                                                                                                                            
    PIAH = z[..., 4]
    PIAV = z[..., 5]
    
    ZH = Zh - PIAH
    ZDR = Zdr - (PIAH-PIAV)
                                                                                                                                             
    return torch.stack([ZH, ZDR, PhiDP], dim=-1)

def Dynamics_allsteps(z_half, N0_range, loc_range, log_scale_range, lookup_tables):
    #print(z_half.shape)
    [table_mu_AH, table_mu_AV, table_mu_kdp, table_mu_ZH, table_mu_Zdr] = lookup_tables
    N0s = torch.sigmoid(z_half[..., 0:1]/(N0_range[1]-N0_range[0]))*(N0_range[1]-N0_range[0])+N0_range[0]
    locs = torch.sigmoid(z_half[..., 1:2]/(loc_range[1]-loc_range[0]))*(loc_range[1]-loc_range[0])+loc_range[0]
    log_scales = torch.sigmoid(z_half[..., 2:3]/(log_scale_range[1]-log_scale_range[0]))*(log_scale_range[1]-log_scale_range[0])+log_scale_range[0]
    
    xp = torch.linspace(-1.2, 1.2, 61, device=z_half.device)
    yp = torch.linspace(-0.2, 0.2, 51, device=z_half.device)
    Zh = torch_interp2d(x=locs, y=log_scales, xp=xp, yp=yp, zp=table_mu_ZH) + 10*N0s
    Zdr = torch_interp2d(x=locs, y=log_scales, xp=xp, yp=yp, zp=table_mu_Zdr)
    kDP = torch_interp2d(x=locs, y=log_scales, xp=xp, yp=yp, zp=table_mu_kdp) * pow(10, N0s)
    AH = torch_interp2d(x=locs, y=log_scales, xp=xp, yp=yp, zp=table_mu_AH) * pow(10, N0s)
    AV = torch_interp2d(x=locs, y=log_scales, xp=xp, yp=yp, zp=table_mu_AV) * pow(10, N0s)
                                                                                                        
    PIAH = torch.cumsum(AH, dim=-2)
    PIAV = torch.cumsum(AV, dim=-2)
    PhiDP = torch.cumsum(kDP, dim=-2)
    return torch.concat([z_half[..., 0:1], z_half[..., 1:2], z_half[..., 2:3], PhiDP, PIAH, PIAV], dim=-1)

def N0(z, N0_range=[0, 5]):
    N0s = torch.sigmoid(z[..., 0:1]/(N0_range[1]-N0_range[0]))*(N0_range[1]-N0_range[0])+N0_range[0]
    return N0s

def loc(z, loc_range):
    locs = torch.sigmoid(z[..., 1:2]/(loc_range[1]-loc_range[0]))*(loc_range[1]-loc_range[0])+loc_range[0]
    return locs

def scale(z, loc_range, log_scale_range):
    locs = torch.sigmoid(z[..., 1:2]/(loc_range[1]-loc_range[0]))*(loc_range[1]-loc_range[0])+loc_range[0]
    log_scale_correctionlist = torch.sigmoid(z[..., 2:3]/(log_scale_range[1]-log_scale_range[0]))*(log_scale_range[1]-log_scale_range[0])+log_scale_range[0]
    log_scale = log_scale_correctionlist-0.8729+0.0291*locs-0.0873*locs**2-0.0442*locs**3-0.0925*locs**4
    scalelist = torch.exp(log_scale)
    return scalelist

def D0(z, loc_range, log_scale_range):
    locs = loc(z, loc_range)
    scales = scale(z, loc_range, log_scale_range)
    return torch.exp(locs+3*scales*scales)

def R_function(z, CR, N0_range=[0, 5], loc_range=[4, 10], log_scale_range=[-1, 1]):
    N0s = torch.sigmoid(z[..., 0:1]/(N0_range[1]-N0_range[0]))*(N0_range[1]-N0_range[0])+N0_range[0]
    locs = torch.sigmoid(z[..., 1:2]/(loc_range[1]-loc_range[0]))*(loc_range[1]-loc_range[0])+loc_range[0]
    log_scales = torch.sigmoid(z[..., 2:3]/(log_scale_range[1]-log_scale_range[0]))*(log_scale_range[1]-loc_range[0])+log_scale_range[0]
    R = pow(10, N0s+CR)*torch.exp(torch.lgamma(4.67+locs))*pow(scales, -(4.67+locs))
    return R


lookup_tables = load_lookups()


In [ ]:
Kumagaya_coord = [-20.469766777364697, 32.122450968084027]
#Kumagaya_coord = [0.0, 0.0] # for visualize

epoch = 0
timestamps = datetime_Kumagaya

lookup_tables = load_lookups()
os.makedirs(f"{OUTDIR}", exist_ok=True)
os.makedirs(f"{OUTDIR}/image", exist_ok=True)
Ndata_per_rank = 340
for rank in range(0, 1):
    for i in range(0, 40):
        idx_time = rank*Ndata_per_rank+i
        #CkDP = torch.load(FOLDER+f"test_epoch{epoch}_iter{i}_CkDP_rank{rank}", map_location="cpu").detach()
        mu_t_list_all = np.load(FOLDER+f"test_rank{rank}_iter{i}_mu_t_list_all.npz")["arr_0"]#_rank{rank}.npz")["arr_0"]
        mu_t_list_all_full = Dynamics_allsteps(z_half=torch.tensor(mu_t_list_all), 
                                               N0_range=[0, 4],
                                               loc_range=[-1.0, 1.0],
                                               log_scale_range=[-0.05, 0.05],
                                               lookup_tables=lookup_tables)
        mu_t_list_all_output = rearrange(mu_t_list_all_full, "a b c -> (a b) c")
        output_flat = observation_DSD_Zh(mu_t_list_all_output,
                                         N0_range=[0, 4],
                                         loc_range=[-1.0, 1.0],
                                         log_scale_range=[-0.05, 0.05],
                                         lookup_tables=lookup_tables)
        output = rearrange(output_flat, "(a b) c -> a b c", a=len(mu_t_list_all)).detach().cpu().numpy()
        timestamp = timestamps[idx_time]
        date = timestamp[2:8]
        obsdata = torch.tensor(np.load(FOLDER+f"test_rank{rank}_iter{i}_obsdata.npz")["arr_0"]).detach().cpu().numpy()
        attenuation = np.load(FOLDER+f"test_rank{rank}_iter{i}_h_attenuation.npz")["arr_0"].reshape(3300, 800)
        print(f"{attenuation=}")
        #obsdata[..., 0] += 8.0
        mask = torch.tensor(np.load(FOLDER+f"test_rank{rank}_iter{i}_maskall.npz")["arr_0"]).detach().cpu().numpy()
        angle_offset = np.loadtxt(FOLDER+f"test_rank{rank}_iter{i}_angle_offset.txt")
        
        modification_radome = np.zeros_like(output)
        modification_radome[:, :, 0] = -attenuation
        modification_short = np.zeros_like(output)
        modification_short[:, :118, 2] = 1.0
        modification_long = np.zeros_like(output)
        modification_long[:, 118:, 2] = 1.0
        model_PhiDP_offset_short = torch.load(FOLDER+f"test_rank{rank}_iter{i}_model_PhiDP_offset_short", map_location="cpu").detach()
        model_PhiDP_offset_long = torch.load(FOLDER+f"test_rank{rank}_iter{i}_model_PhiDP_offset_long", map_location="cpu").detach()
        
        output = output + modification_radome + model_PhiDP_offset_short.numpy()*modification_short+model_PhiDP_offset_long.numpy()*modification_long

        kDP = kDP_function(mu_t_list_all_full,
                           N0_range=[0, 4],
                           loc_range=[-1.0, 1.0],
                           log_scale_range=[-0.05, 0.05],
                           lookup_tables=lookup_tables)
        N0s = N0(mu_t_list_all_full, N0_range=[0, 4])
        locs = loc(mu_t_list_all_full, loc_range=[-1.0, 1.0])
        D0s = D0(mu_t_list_all_full, loc_range=[-1.0, 1.0], log_scale_range=[-0.05, 0.05])
        for el in range(5, 6):
            elevation = el*0.5
            X, Y = output_coordinate(angle_offset, elevation)
            # mask for mu, D0
            mask2 = obsdata[el*300:(el+1)*300, :, 0] > 3

            fig = plt.figure(figsize=(12, 10))

            ax1 = fig.add_subplot(3,4,1)
            ax1.set_xlim(-65, 65)
            ax1.set_aspect('equal', 'box')
            ax1.scatter(X, Y, c=output[el*300:(el+1)*300, :, 0], vmin=10, vmax=45, s=1)
            ax1.scatter(Kumagaya_coord[0], Kumagaya_coord[1], marker="*", c="k", s=50)
            ax1.set_title(f"ZH, model")
            #ax1.set_xlabel("ZH")
            #plt.savefig(f"obsdata_visualize/{el}/model_ZH_time{timestamp}_el{el}")
            #plt.close()
            #plt.figure(figsize=(7, 7))
            ax2 = fig.add_subplot(3,4,2)
            ax2.set_xlim(-65, 65)
            ax2.set_aspect('equal', 'box')
            ax2.scatter(X, Y, c=obsdata[el*300:(el+1)*300, :, 0], vmin=10, vmax=45, s=1)
            ax2.scatter(Kumagaya_coord[0], Kumagaya_coord[1], marker="*", c="k", s=50)
            ax2.set_title(f"ZH, obsdata")

            ax3 = fig.add_subplot(3,4,3)
            ax3.set_xlim(-65, 65)
            ax3.set_aspect('equal', 'box')
            ax3.scatter(X, Y, c=mu_t_list_all_full[el*300:(el+1)*300, :, 4].detach(), vmin=0, vmax=30, s=1)
            ax3.scatter(Kumagaya_coord[0], Kumagaya_coord[1], marker="*", c="k", s=50)
            ax3.set_title(f"PIAH")
            fig.suptitle(f"{date=}, time={timestamp}, el={elevation:.1f}deg")

            ax3 = fig.add_subplot(3,4,4)
            ax3.set_xlim(-65, 65)
            ax3.set_aspect('equal', 'box')
            ax3.scatter(X, Y, c=N0s[el*300:(el+1)*300, :].detach(), vmin=3, vmax=5, s=1)
            ax3.scatter(Kumagaya_coord[0], Kumagaya_coord[1], marker="*", c="k", s=50)
            ax3.set_title(f"N0")
            #fig.suptitle(f"time={timestamp}, {el=}")

            ax4 = fig.add_subplot(3,4,5)
            ax4.set_xlim(-65, 65)
            ax4.set_aspect('equal', 'box')
            ax4.scatter(X, Y, c=output[el*300:(el+1)*300, :, 1]*mask[el*300:(el+1)*300, :, 1], vmin=0, vmax=2, s=1)
            ax4.scatter(Kumagaya_coord[0], Kumagaya_coord[1], marker="*", c="k", s=50)
            ax4.set_title(f"Zdr, model")
            #ax4.set_xlabel("Zdr")
            #plt.savefig(f"obsdata_visualize/{el}/model_ZH_time{timestamp}_el{el}")
            #plt.close()
            #plt.figure(figsize=(7, 7))
            ax5 = fig.add_subplot(3,4,6)
            ax5.set_xlim(-65, 65)
            ax5.set_aspect('equal', 'box')
            ax5.scatter(X, Y, c=obsdata[el*300:(el+1)*300, :, 1]*mask[el*300:(el+1)*300, :, 1], vmin=0, vmax=2, s=1)
            ax5.scatter(Kumagaya_coord[0], Kumagaya_coord[1], marker="*", c="k", s=50)
            ax5.set_title(f"Zdr, obsdata")

            ax6 = fig.add_subplot(3,4,7)
            ax6.set_xlim(-65, 65)
            ax6.set_aspect('equal', 'box')
            ax6.scatter(X, Y, c=mu_t_list_all_full[el*300:(el+1)*300, :, 5].detach(), vmin=0, vmax=30, s=1)
            ax6.scatter(Kumagaya_coord[0], Kumagaya_coord[1], marker="*", c="k", s=50)
            ax6.set_title(f"PIAV")

            ax3 = fig.add_subplot(3,4,8)
            ax3.set_xlim(-65, 65)
            ax3.set_aspect('equal', 'box')
            ax3.scatter(X, Y, c=locs[el*300:(el+1)*300, :, 0].detach()*mask2, vmin=0, vmax=8, s=1)
            ax3.scatter(Kumagaya_coord[0], Kumagaya_coord[1], marker="*", c="k", s=50)
            ax3.set_title(f"loc")
            #fig.suptitle(f"time={timestamp}, {el=}")

            ax7 = fig.add_subplot(3,4,9)
            ax7.set_xlim(-65, 65)
            ax7.set_aspect('equal', 'box')
            ax7.scatter(X, Y, c=output[el*300:(el+1)*300, :, 2]*mask[el*300:(el+1)*300, :, 2], vmin=0, vmax=60, s=1)
            ax7.scatter(Kumagaya_coord[0], Kumagaya_coord[1], marker="*", c="k", s=50)
            ax7.set_title(f"PhiDP, model")
            #plt.savefig(f"obsdata_visualize/{el}/model_ZH_time{timestamp}_el{el}")
            #plt.close()
            #plt.figure(figsize=(7, 7))
            ax8 = fig.add_subplot(3,4,10)
            ax8.set_xlim(-65, 65)
            ax8.set_aspect('equal', 'box')
            ax8.scatter(X, Y, c=obsdata[el*300:(el+1)*300, :, 2]*mask[el*300:(el+1)*300, :, 2], vmin=0, vmax=60, s=1)
            ax8.scatter(Kumagaya_coord[0], Kumagaya_coord[1], marker="*", c="k", s=50)
            ax8.set_title(f"PhiDP, obsdata")

            ax9 = fig.add_subplot(3,4,11)
            ax9.set_xlim(-65, 65)
            ax9.set_aspect('equal', 'box')
            ax9.scatter(X, Y, c=kDP[el*300:(el+1)*300, :].detach(), vmin=0, vmax=1.0, s=1)
            ax9.scatter(Kumagaya_coord[0], Kumagaya_coord[1], marker="*", c="k", s=50)
            ax9.set_title(f"kDP, model")
            #fig.suptitle(f"{date=}, time={timestamp}, {el=}")

            ax3 = fig.add_subplot(3,4,12)
            ax3.set_xlim(-65, 65)
            ax3.set_aspect('equal', 'box')
            ax3.scatter(X, Y, c=D0s[el*300:(el+1)*300, :, 0].detach()*mask2, vmin=1, vmax=3, s=1)
            ax3.scatter(Kumagaya_coord[0], Kumagaya_coord[1], marker="*", c="k", s=50)
            ax3.set_title(f"D0")
            #fig.suptitle(f"time={timestamp}, {el=}")

            fig.savefig(f"{OUTDIR}/image/obsdata_ZH_data{i}_date{date}_time{timestamp}_el{el}")
            plt.close()